# 📊 Model Evaluation — GPU Cooling Digital Twin

**Purpose**: Evaluate the trained model and compare with baselines.

> ⚠️ **SYNTHETIC DATA** — All metrics below are from synthetic telemetry evaluation.

In [ ]:
!pip install lightgbm scikit-learn joblib pandas numpy matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import joblib
import json
import time
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.style.use('dark_background')
sns.set_palette('husl')
print('Libraries loaded.')

In [ ]:
# === LOAD MODEL AND DATA ===

try:
    from google.colab import drive
    drive.mount('/content/drive')
    model = joblib.load('/content/drive/MyDrive/thermal_model.joblib')
    df = pd.read_parquet('/content/drive/MyDrive/gpu_cooling_dataset.parquet')
    with open('/content/drive/MyDrive/model_metadata.json') as f:
        metadata = json.load(f)
except:
    model = joblib.load('thermal_model.joblib')
    df = pd.read_parquet('gpu_cooling_dataset.parquet')
    with open('model_metadata.json') as f:
        metadata = json.load(f)

feature_cols = metadata['feature_names']
target_cols = metadata['target_variables']

print(f'Model: {metadata["model_type"]}')
print(f'Features: {len(feature_cols)}')
print(f'Dataset: {df.shape}')

In [ ]:
# === PREPARE TEST SET ===

X = df[feature_cols].values
y = df[target_cols].values

n = len(X)
val_end = int(n * 0.85)
X_test, y_test = X[val_end:], y[val_end:]

y_pred = model.predict(X_test)

print(f'Test set: {X_test.shape[0]:,} samples')

In [ ]:
# === NAIVE BASELINE: LINEAR EXTRAPOLATION ===

# Baseline: predict future temperature as current + slope * horizon
temp_col_idx = feature_cols.index('gpu_temperature')
slope_col_idx = feature_cols.index('temperature_slope_5')

baseline_temp_pred = X_test[:, temp_col_idx] + X_test[:, slope_col_idx] * 30

power_col_idx = feature_cols.index('gpu_power')
power_slope_idx = feature_cols.index('power_slope_5')
baseline_power_pred = X_test[:, power_col_idx] + X_test[:, power_slope_idx] * 30

# Baseline metrics
bl_temp_mae = mean_absolute_error(y_test[:, 0], baseline_temp_pred)
bl_temp_rmse = np.sqrt(mean_squared_error(y_test[:, 0], baseline_temp_pred))
bl_temp_r2 = r2_score(y_test[:, 0], baseline_temp_pred)

bl_power_mae = mean_absolute_error(y_test[:, 1], baseline_power_pred)
bl_power_rmse = np.sqrt(mean_squared_error(y_test[:, 1], baseline_power_pred))
bl_power_r2 = r2_score(y_test[:, 1], baseline_power_pred)

# ML model metrics
ml_temp_mae = mean_absolute_error(y_test[:, 0], y_pred[:, 0])
ml_temp_rmse = np.sqrt(mean_squared_error(y_test[:, 0], y_pred[:, 0]))
ml_temp_r2 = r2_score(y_test[:, 0], y_pred[:, 0])

ml_power_mae = mean_absolute_error(y_test[:, 1], y_pred[:, 1])
ml_power_rmse = np.sqrt(mean_squared_error(y_test[:, 1], y_pred[:, 1]))
ml_power_r2 = r2_score(y_test[:, 1], y_pred[:, 1])

In [ ]:
# === COMPARISON TABLE ===

comparison = pd.DataFrame({
    'Metric': ['Temp MAE (°C)', 'Temp RMSE (°C)', 'Temp R²',
               'Power MAE (W)', 'Power RMSE (W)', 'Power R²'],
    'Linear Baseline': [f'{bl_temp_mae:.3f}', f'{bl_temp_rmse:.3f}', f'{bl_temp_r2:.4f}',
                        f'{bl_power_mae:.3f}', f'{bl_power_rmse:.3f}', f'{bl_power_r2:.4f}'],
    'LightGBM Model': [f'{ml_temp_mae:.3f}', f'{ml_temp_rmse:.3f}', f'{ml_temp_r2:.4f}',
                       f'{ml_power_mae:.3f}', f'{ml_power_rmse:.3f}', f'{ml_power_r2:.4f}']
})

print('\n=== MODEL COMPARISON (SYNTHETIC DATA) ===')
print(comparison.to_string(index=False))

In [ ]:
# === VISUALIZATION: ACTUAL VS PREDICTED ===

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Model Evaluation — Synthetic Data', fontsize=16, color='white')

# Temperature: time series
window = slice(0, 1000)
axes[0, 0].plot(y_test[window, 0], label='Actual', alpha=0.8, color='#00e5ff')
axes[0, 0].plot(y_pred[window, 0], label='ML Predicted', alpha=0.8, color='#ff6b6b')
axes[0, 0].plot(baseline_temp_pred[window], label='Linear Baseline', alpha=0.5, color='#888', linestyle='--')
axes[0, 0].set_title('Temperature Prediction (30s Horizon)')
axes[0, 0].set_ylabel('Temperature (°C)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.2)

# Temperature: scatter
axes[0, 1].scatter(y_test[::50, 0], y_pred[::50, 0], alpha=0.3, s=5, color='#00e5ff')
axes[0, 1].plot([y_test[:, 0].min(), y_test[:, 0].max()],
               [y_test[:, 0].min(), y_test[:, 0].max()], 'r--', alpha=0.5)
axes[0, 1].set_title(f'Temperature: Actual vs Predicted (R²={ml_temp_r2:.4f})')
axes[0, 1].set_xlabel('Actual (°C)')
axes[0, 1].set_ylabel('Predicted (°C)')
axes[0, 1].grid(alpha=0.2)

# Power: time series
axes[1, 0].plot(y_test[window, 1], label='Actual', alpha=0.8, color='#00e5ff')
axes[1, 0].plot(y_pred[window, 1], label='ML Predicted', alpha=0.8, color='#ff6b6b')
axes[1, 0].set_title('Power Prediction (30s Horizon)')
axes[1, 0].set_ylabel('Power (W)')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.2)

# Error distribution
temp_errors = y_test[:, 0] - y_pred[:, 0]
axes[1, 1].hist(temp_errors, bins=100, color='#00e5ff', alpha=0.7)
axes[1, 1].set_title(f'Temperature Prediction Error Distribution')
axes[1, 1].set_xlabel('Error (°C)')
axes[1, 1].axvline(0, color='red', linestyle='--', alpha=0.5)
axes[1, 1].grid(alpha=0.2)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()
print('Evaluation plots saved.')

In [ ]:
# === INFERENCE LATENCY BENCHMARK ===

latencies = []
for _ in range(1000):
    sample = X_test[np.random.randint(len(X_test))].reshape(1, -1)
    start = time.perf_counter()
    _ = model.predict(sample)
    latencies.append((time.perf_counter() - start) * 1000)

print(f'\n=== INFERENCE LATENCY ===')
print(f'Mean:   {np.mean(latencies):.3f} ms')
print(f'Median: {np.median(latencies):.3f} ms')
print(f'P95:    {np.percentile(latencies, 95):.3f} ms')
print(f'P99:    {np.percentile(latencies, 99):.3f} ms')
print(f'\nSuitable for real-time inference at 1Hz: {"YES" if np.mean(latencies) < 100 else "NO"}')

In [ ]:
# === FEATURE IMPORTANCE ===

importances = model.estimators_[0].feature_importances_
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': importances
}).sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='#00e5ff')
ax.set_title('Feature Importance (Temperature Prediction)')
ax.set_xlabel('Importance')
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.show()

In [ ]:
# === FINAL SUMMARY ===

print('\n' + '='*60)
print('EVALUATION SUMMARY — SYNTHETIC DATA')
print('='*60)
print(f'Model: {metadata["model_type"]}')
print(f'Prediction Horizon: {metadata["prediction_horizon_seconds"]}s')
print(f'\nTemperature Prediction:')
print(f'  MAE:  {ml_temp_mae:.3f}°C (baseline: {bl_temp_mae:.3f}°C)')
print(f'  RMSE: {ml_temp_rmse:.3f}°C (baseline: {bl_temp_rmse:.3f}°C)')
print(f'  R²:   {ml_temp_r2:.4f} (baseline: {bl_temp_r2:.4f})')
print(f'\nPower Prediction:')
print(f'  MAE:  {ml_power_mae:.3f}W (baseline: {bl_power_mae:.3f}W)')
print(f'  RMSE: {ml_power_rmse:.3f}W (baseline: {bl_power_rmse:.3f}W)')
print(f'  R²:   {ml_power_r2:.4f} (baseline: {bl_power_r2:.4f})')
print(f'\nInference: {np.mean(latencies):.2f}ms avg')
print(f'\n⚠️ All metrics are from SYNTHETIC data evaluation.')
print('='*60)